# 03 — Attribute Library

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

The candidate attribute list that goes to classing. The registry in `scorelab.attributes` holds every raw bureau field the keep-list allows plus a handful of derived ratios, each with a family, a plain-language meaning (used verbatim in reason codes) and an expected direction of risk (used as the monotone constraint). You decide which enter, and you own the meanings.

Decisions: **C — attribute families and exclusions**, **D — derived attributes to keep or add**, **E — the candidate list** written to `models/candidate_attributes.json`.

In [ ]:
# Leakage control: every source column of every attribute must be a keep-list input (raises otherwise)
A.leakage_check()
reg = A.registry_table(); display(reg)
reg.to_csv(config.DOCS_DIR / "exhibits" / "attribute_registry.csv", index=False)

In [ ]:
# Build the attributes on the modelling frame
frame = load_frame()
tr = frame[frame.split == "train"]
names = A.attribute_names()
print(frame.shape, "train", tr.shape, "attributes", len(names))
gaps = json.loads((MODELS / "audit_decisions.json").read_text(encoding="utf-8"))["structural_gap_fields"] if (MODELS / "audit_decisions.json").exists() else []
print("structural-gap fields from NB01:", gaps)

In [ ]:
# Evidence — availability by vintage (attributes, after derivation)
display(A.availability_by_vintage(frame, names))

In [ ]:
# Evidence — univariate rank power on train, with the expected vs empirical direction
screen = A.univariate_screen(tr, names)
display(screen)
screen.to_csv(config.DOCS_DIR / "exhibits" / "univariate_screen_train.csv", index=False)
print("direction conflicts (expected vs empirical):")
display(screen[(screen.expected_direction != "no prior") & (screen.expected_direction != screen.empirical_direction)])

In [ ]:
# Evidence — redundancy: pairs of attributes with |Spearman| > 0.7 on train
corr = tr[names].corr(method="spearman")
pairs = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack().rename("rho").reset_index()
         .query("abs(rho) > 0.7").sort_values("rho", key=abs, ascending=False))
display(pairs)

### Decision C — Families and exclusions

Which families carry the card (delinquency, inquiries, utilisation, credit age, file depth)? Anything to exclude for business or compliance reasons even if predictive? The structural-gap fields from NB01 are dropped by default.

**Your reasoning:**

_(write here)_

### Decision D — Derived attributes

The registry derives `credit_age_months`, `new_trades_share_24m`, `revol_bal_to_limit`, `bal_ex_mort_to_limit`, `any_derog_flag`, `ever_delinquent_flag`. Keep, drop, or add (edit `scorelab/attributes.py` and re-run)?

**Your reasoning:**

_(write here)_

### Decision E — The candidate list

Default below: available from 2012, not a structural-gap field, single-variable Gini at least 0.02 on train, minus your exclusions. Edit `EXCLUDE` and the threshold, then run.

**Your reasoning:**

_(write here)_

In [ ]:
EXCLUDE = []                      # your business / compliance exclusions, by attribute name
MIN_GINI = 0.02
ok = screen[(screen.gini_abs >= MIN_GINI)].attribute.tolist()
candidates = [n for n in A.attribute_names(min_available_from=2012)
              if n in ok and n not in EXCLUDE and not set(A.REGISTRY_BY_NAME[n].sources) & set(gaps)]
(MODELS / "candidate_attributes.json").write_text(json.dumps({"candidates": candidates, "exclude": EXCLUDE, "min_gini": MIN_GINI}, indent=1), encoding="utf-8")
print(len(candidates), "candidates saved to", MODELS / "candidate_attributes.json"); print(candidates)